# WeakTr on Kaggle

Import this notebook (or clone the repository), attach your **private input dataset**, and run cells in order. The input dataset contains the prepared `weaktr-kaggle-inputs.zip` extracted by Kaggle. To resume, also attach the extracted `weaktr-resume.zip` from a previous run. No weights, images or generated masks belong in Git.

The configuration below is the only place to choose fresh/resume and optional pilot behavior. `PILOT = False` runs the full official pipeline; `True` runs both synthetic GPU checks and one Phase-1 epoch, then stops. The pilot preserves the full 60-epoch scheduler. Setup and data checks use CPU; training requires one Kaggle GPU. You may run setup with the accelerator disabled to preserve GPU allocation.

Kaggle/CUDA compatibility and final scores remain unverified until actually run. Budget ceiling: the user-reported 60 Kaggle hours, with 12 reserved. A pilot can supply timing for the forecast; a full run never shortens the schedules automatically.

VS Code is also supported: select the Kaggle remote Jupyter server as this notebook's kernel. The notebook file may stay on Windows; execution, cloned code, datasets and outputs all live on Kaggle. Local edits to separate Python files must be pushed to the branch before updating the remote checkout.

In [ ]:
# 1. Configuration — normally only PILOT or RESUME_FROM needs changing.
from pathlib import Path
import os, subprocess, sys
if not Path("/kaggle").is_dir():
    raise RuntimeError("Select the Kaggle remote Jupyter kernel (or run in Kaggle) before executing this notebook.")
print("Execution kernel:", sys.executable, "on", sys.platform)
REPO = Path('/kaggle/working/WeakTr')
WORK = Path('/kaggle/working')
INPUTS = Path('/kaggle/input')  # auto-finds the attached private input dataset
RESUME_FROM = 'auto'          # auto | none (fresh) | exact saved-run directory
PILOT = False                # True: optional first-run smoke + one-epoch pilot

# Clone automatically when this notebook is imported into an empty Kaggle runtime.
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', 'reproduction/voc-deit-small',
                    'https://github.com/ntmphu/WeakTr.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin',
                    'reproduction/voc-deit-small'], check=True)

def workflow(stage):
    subprocess.run([sys.executable, str(REPO / 'reproduction/kaggle.py'), stage,
                    '--inputs', str(INPUTS), '--work', str(WORK),
                    '--resume-from', RESUME_FROM], cwd=REPO, check=True)
print({'repository': str(REPO), 'inputs': str(INPUTS), 'resume': RESUME_FROM,
       'mode': 'pilot' if PILOT else 'full pipeline'})

## 2. Setup and validate inputs

Run this cell on the first session and after a runtime restart. It discovers the input dataset, copies and verifies the files onto local disk, installs an isolated Python 3.10 environment, checks both trainer imports/CRF, runs CPU tests, and validates the real dataset. An attached saved run is restored with hash verification. Existing local checkpoints take precedence in automatic mode. No GPU training is performed here.

If multiple saved runs are attached, set `RESUME_FROM` to the one you want. Keep the input dataset attached in every session. Re-running setup reuses files that match the manifest.

In [ ]:
workflow('prepare')

## 3. Optional pilot

With `PILOT = True`, this cell checks both models' GPU forward/backward paths using synthetic inputs. The next cell then runs one complete Phase-1 epoch, including official classification/CAM selection. These checks are distinct from GPU recovery equivalence and author-checkpoint evaluation; those remain measured acceptance tasks described in `docs/KAGGLE.md`.

With `PILOT = False`, this cell is skipped. Run subsequent cells with one GPU enabled.

In [ ]:
if PILOT:
    workflow('smoke')
else:
    print('Optional smoke/pilot mode is off; full official schedules will run.')

## 4. Phase 1

Uses ImageNet initialization for a fresh run, otherwise resumes the verified full snapshot. Full schedule: 60 epochs, effective batch 64, supplied LR 0.0006 (scaled to 0.000075), reduction 8, max pooling and weight decay 0.03. Selection uses ground-truth masks on the official train list. A finished checkpoint skips completed training epochs.

After training, the notebook creates `/kaggle/working/weaktr-resume.zip` for download or a private Kaggle dataset. To stop earlier without interrupting an epoch, use the pilot option or the CLI's `--stop-after-epoch`; a hard interruption replays the unfinished epoch.

In [ ]:
workflow('phase1-pilot' if PILOT else 'phase1')
if PILOT:
    print('PILOT COMPLETE. Set PILOT = False, rerun configuration, then continue from Phase 1; its checkpoint resumes automatically.')

## 5. Reproduction CAMs and pseudo-labels

Skipped in pilot mode. Generate training CAMs at scales 1.0/1.2 from the reproduction's best CAM checkpoint, then CRF masks at threshold 43 (0.43). A resumed Phase-2 run reuses its saved, complete pseudo-label collection. Author reference masks are never substituted.

In [ ]:
if not PILOT:
    workflow('cams')
    workflow('pseudo')
    workflow('preserve')
else:
    print('Skipped after the one-epoch pilot.')

## 6. Phase 2

Skipped in pilot mode. Runs or resumes the original 100-epoch schedule: batch 4, SGD/poly LR 0.0001, encoder multiplier 0.1, GC 1.2/120, resize 520, crop/window 480, stride 320, evaluation every epoch. Training annotations are the generated pseudo-labels; validation uses GT.

In [ ]:
if not PILOT:
    workflow('phase2')

## 7. Final evaluation

Skipped in pilot mode. Preserves upstream multiscale evaluation with the two decoder outputs combined at weight 0.8, followed by CRF. Raw per-class scores are in `weaktr-artifacts/phase2/scores_ms.yml`; post-CRF scores are in the stage log. Preserve raw and post-CRF metrics separately. The 74.0% reference is a target, not a promised result.

In [ ]:
if not PILOT:
    workflow('evaluate')
    workflow('crf')
    workflow('preserve')

## Save and resume the next session

Download `/kaggle/working/weaktr-resume.zip` or preserve its folder in saved notebook outputs. It includes full checkpoints, best checkpoints, configs, logs and generated pseudo-labels; intermediate CAM/probability collections are excluded. Upload the ZIP as another **private Kaggle dataset** and let Kaggle extract it. Attach both the original inputs and this saved run to the next notebook. Keep `RESUME_FROM = 'auto'`, rerun setup, then run the remaining cells. Hashes are checked before checkpoints are restored.

A live runtime's disk alone is not a backup. Export again after any additional completed training if needed:

In [ ]:
# Optional manual export after an additional completed stage:
# workflow('preserve')
print('Latest resumable archive:', WORK / 'weaktr-resume.zip')